# 外部知识 Backlog v1

目标：把 arXiv、paper、GitHub、X/Twitter 等外部知识从简单抓取队列升级为可测试 backlog。这里不保存第三方全文，只保存 URL、自有摘要、license gate、目标 `src/` 接口和 research 验证计划。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import KnowledgeSource, build_knowledge_backlog, write_knowledge_backlog

OUTPUT_DIR = REPO_ROOT / "research" / "knowledge_backlog_v1"
sources = [
    KnowledgeSource(
        source_id="arxiv:2504.18600",
        source_type="arxiv",
        title="QuantBench: Benchmarking AI Methods for Quantitative Investment",
        url="https://arxiv.org/abs/2504.18600",
        summary="QuantBench 作为模型矩阵和研究流程参考，重点转化为统一 registry、fit/predict、backtest、analysis 和 promotion 证据链。",
        license_note="只记录公开元数据和自有摘要；不复制论文全文。",
    ),
    KnowledgeSource(
        source_id="arxiv:2211.14730",
        source_type="arxiv",
        title="A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
        url="https://arxiv.org/abs/2211.14730",
        summary="PatchTST 的 patch 和 channel-independent 思路可用于替换 patch_tst_mse proxy，并验证长窗口 SOXL 支持特征。",
        license_note="只记录 arXiv 元数据和自有摘要；实现前单独检查代码许可。",
    ),
    KnowledgeSource(
        source_id="arxiv:2205.13504",
        source_type="paper",
        title="Are Transformers Effective for Time Series Forecasting?",
        url="https://arxiv.org/abs/2205.13504",
        summary="DLinear/decomposition-linear baseline 可用于替换 dlinear_mse proxy，并作为复杂 transformer 模型的强基线。",
        license_note="只记录 arXiv 元数据和自有摘要；实现前单独检查代码许可。",
    ),
    KnowledgeSource(
        source_id="github:microsoft/qlib",
        source_type="github",
        title="Qlib AI-oriented quantitative investment platform",
        url="https://github.com/microsoft/qlib",
        summary="Qlib recorder/analyzer 工作流可继续扩展 src/analysis 的 recorder 解析、risk analysis、dashboard manifest export。",
        license_note="GitHub 源码实现前必须检查 upstream license；当前只保留研究入口。",
    ),
    KnowledgeSource(
        source_id="paper:rsr-stock-ranking",
        source_type="paper",
        title="Relational stock ranking model references",
        url="https://arxiv.org/search/?query=relational+stock+ranking+RSR&searchtype=all",
        summary="RSR/关系建模线索用于替换 rsr_mse proxy，优先验证关系强度、行业/ETF 支持图和 regime 稳定性。",
        license_note="检索入口和自有摘要；具体论文和代码进入实现前需逐项复核许可。",
    ),
    KnowledgeSource(
        source_id="paper:temporal-hypergraph-stock",
        source_type="paper",
        title="Temporal hypergraph stock prediction references",
        url="https://arxiv.org/search/?query=temporal+hypergraph+stock+prediction&searchtype=all",
        summary="THGNN/STHGCN 类线索用于替换 temporal_hypergraph proxy，重点验证 hyperedge 定义和压力期过拟合。",
        license_note="检索入口和自有摘要；具体论文和代码进入实现前需逐项复核许可。",
    ),
    KnowledgeSource(
        source_id="x:regime-aware-quant-review",
        source_type="x",
        title="Regime-aware quant model review social search",
        url="https://x.com/search?q=regime%20aware%20quant%20evaluation",
        summary="社交媒体只作为发散来源：把波动、回撤、趋势 regime 的 review 思路转成 src/analysis 可测试分段。",
        license_note="不复制帖子正文；只保留入口和自有摘要。",
    ),
]

write_result = write_knowledge_backlog(OUTPUT_DIR, sources)
backlog = build_knowledge_backlog(sources)
summary = {
    "schema_version": "research.knowledge_backlog_v1",
    "status": "completed",
    "written": write_result["written"],
    "deduped": write_result["deduped"],
    "source_count": len(backlog),
    "target_subsystems": sorted({item["target_subsystem"] for item in backlog}),
    "candidate_types": sorted({item["candidate_type"] for item in backlog}),
    "top_candidates": [item["candidate_name"] for item in backlog[:5]],
    "proxy_replacement_candidates": [
        item["candidate_name"] for item in backlog if item["candidate_type"] == "model_replacement"
    ],
    "artifacts": {
        "knowledge_backlog": "knowledge_backlog.json",
        "knowledge_backlog_csv": "knowledge_backlog.csv",
        "knowledge_backlog_review": "knowledge_backlog_review.md",
    },
}
(OUTPUT_DIR / "notebook_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


{'schema_version': 'research.knowledge_backlog_v1',
 'status': 'completed',
 'written': 7,
 'deduped': 0,
 'source_count': 7,
 'target_subsystems': ['src/analysis', 'src/model'],
 'candidate_types': ['analysis_extension',
  'method_signal',
  'model_matrix_reference',
  'model_replacement'],
 'top_candidates': ['dlinear_mse',
  'patch_tst_mse',
  'rsr_mse',
  'thgnn_mse',
  'quantbench_model_matrix'],
 'proxy_replacement_candidates': ['dlinear_mse',
  'patch_tst_mse',
  'rsr_mse',
  'thgnn_mse'],
 'artifacts': {'knowledge_backlog': 'knowledge_backlog.json',
  'knowledge_backlog_csv': 'knowledge_backlog.csv',
  'knowledge_backlog_review': 'knowledge_backlog_review.md'}}

## 中文结论

- 做了什么：把 arXiv、paper、GitHub、X/Twitter 搜索入口整理成 `knowledge_backlog.v1`，每条记录包含目标 `src/` 接口、license gate、验收标准和是否只作为 hypothesis。
- 核心指标：写入 7 条 backlog，去重 0 条；目标覆盖 `src/model` 和 `src/analysis`；优先级前列是 QuantBench、PatchTST、DLinear 等可用于替换 proxy 或增强模型矩阵的来源。
- regime 差异：X/Twitter 和 Qlib 来源只作为 regime/window review 方法线索，必须通过 `src/analysis` 的分段回测、failure mode、model review briefing 验证。
- 是否晋升：否；这是 research backlog，不是 production 方案。任何模型替换进入 production 前都必须完成 notebook、pytest、manifest、回测和中文结论。
- 风险：外部来源存在 license、复现性、数据泄露、实现漂移和社交媒体噪声风险；未通过 license gate 前不能 vendoring 代码，不能写入 skill 作为项目状态。
- 下一步：按 `priority_rank` 建立单独 research notebook，优先验证 `patch_tst_mse`、`dlinear_mse`、`rsr_mse` 和 temporal hypergraph proxy 的替换路径。